## Imports

In [2]:
%load_ext autoreload
%autoreload 2

import logging

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore

In [3]:
import helpers.hnotebook as hnotebo

import helpers.htutorial as ut
# import L08_04_08_difference_in_difference_utils as mtl

ut.config_notebook()

# Initialize logger.
logging.basicConfig(level=logging.INFO)
_LOG = logging.getLogger(__name__)
hnotebo.set_logger_to_print(_LOG)
hnotebo.set_all_loggers_to_print()

pymc is not installed
arviz is not installed
preliz is not installed
sns is not installed


Python 3.12.13
Linux 3a71c0a7bf16 6.12.67-linuxkit #1 SMP Sun Jan 25 02:26:28 UTC 2026 aarch64 GNU/Linux


In [4]:
# import warnings

# import helpers.hmodule as hmodule
# from lightgbm import LGBMRegressor
# import fklearn.causal.validation.curves
# import fklearn.causal.validation.auc

# warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")
# warnings.filterwarnings(
#    "ignore",
#    message="X does not have valid feature names",
#    category=UserWarning,
# )
# logging.getLogger("lightgbm").setLevel(logging.ERROR)

# hmodule.install_module_if_not_present(
#    ["lightgbm", "fklearn"],
#    use_activate=True,
#    use_sudo=False,
#    venv_path="/opt/venv",
# )

# Load data

In [5]:
dir_name = "L08_data"
#!ls $dir_name

out_dir_name = "figures/"

In [9]:
mkt_data = pd.read_csv(f"{dir_name}/short_offline_mkt_south.csv").astype(
    {"date": "datetime64[ns]"}
)
print("mkt_data=", mkt_data.shape)
display(mkt_data.head())

# Marketing data in a panel format.
# - Each line is a (`date`, `city`)
# - The outcome to predict is `downloads`
# - `treated` is the indicator of the intervention
# - `tau` is the treatment effect

mkt_data= (1632, 7)


,date,city,region,treated,tau,downloads,post
0,2021-05-01,5,S,0,0.0,51.0,0
1,2021-05-02,5,S,0,0.0,51.0,0
2,2021-05-03,5,S,0,0.0,51.0,0
3,2021-05-04,5,S,0,0.0,50.0,0
4,2021-05-05,5,S,0,0.0,49.0,0


In [11]:
# Compute pre- and post-intervention period.
(
    mkt_data.assign(w=lambda d: d["treated"] * d["post"])
    .groupby(["w"])
    .agg({"date": ["min", "max"]})
)

date           
         min        max
w                      
0 2021-05-01 2021-06-01
1 2021-05-15 2021-06-01

In [ ]:
## Canonical DiD

In [12]:
did_data = mkt_data.groupby(["treated", "post"]).agg(
    {"downloads": "mean", "date": "min"}
)

did_data

downloads       date
treated post                      
0       0     50.335034 2021-05-01
        1     50.556878 2021-05-15
1       0     50.944444 2021-05-01
        1     51.858025 2021-05-15

In [13]:
y0_est = (
    did_data.loc[1].loc[0, "downloads"]  # treated baseline
    # control evolution
    + did_data.loc[0].diff().loc[1, "downloads"]
)

att = did_data.loc[1].loc[1, "downloads"] - y0_est
att

np.float64(0.6917359536407233)

In [14]:
mkt_data.query("post==1").query("treated==1")["tau"].mean()

np.float64(0.7660316402518457)